In [ ]:
import os
import simplnx as nx
import orientationanalysis as nxor
root_path = "/home/kaiobach/Research/hu_hu_hu/sprint_vitesh/SCORE_modified/examples/amag"
file_name = "Microstructure_for_paper_tensionX_remeshed_increment_6000.dream3d"

In [ ]:
def check_filter_result(filter: nx.IFilter, result: nx.IFilter.ExecuteResult) -> None:
    """
    This function will check the `result` for any errors. If errors do exist then a
    `RuntimeError` will be thrown. Your own code to modify this to return something
    else that doesn't just stop your script in its tracks.
    """
    if len(result.warnings) != 0:
        for w in result.warnings:
            print(f'Warning: ({w.code}) {w.message}')

    has_errors = len(result.errors) != 0
    if has_errors:
        for err in result.errors:
            print(f'Error: ({err.code}) {err.message}')
            raise RuntimeError(result)
    else:
        print(f"{filter.name()} :: No errors running the filter")

In [ ]:
ds = nx.DataStructure()

In [ ]:
import_data = nx.Dream3dImportParameter.ImportData(
    file_path=f"{root_path}/simulation/{file_name}",
    path_import_policy=nx.Dream3dImportParameter.PathImportPolicy.All
)
result = nx.ReadDREAM3DFilter.execute(
    data_structure=ds,
    import_data_object=import_data
)
check_filter_result(
    nx.ReadDREAM3DFilter(),
    result
)
print(ds.hierarchy_to_str())

In [ ]:
# https://www.dream3d.io/python_docs/OrientationAnalysis.html#OrientationAnalysis.ConvertOrientationsFilter
# https://www.dream3d.io/nx_reference_manual/html/OrientationAnalysis/ConvertOrientationsFilter.html
result = nxor.ConvertOrientationsFilter.execute(
    data_structure=ds,
    input_orientation_array_path=nx.DataPath("SyntheticVolumeDataContainer/CellData/Eulers"),
    input_representation_index=0,
    output_orientation_array_name="Quats",
    output_representation_index=2,
)
check_filter_result(
    nxor.ConvertOrientationsFilter(),
    result
)

In [ ]:
# WIP
result = nxor.EBSDSegmentFeaturesFilter.execute(
    data_structure=ds,
    active_array_name="Active",  
    # cell_feature_attribute_matrix_name (nx.DataObjectNameParameter) – The name of the created cell feature attribute matrix
    # cell_mask_array_path (nx.ArraySelectionParameter) – Path to the data array that specifies if the Cell is to be counted in the algorithm
    cell_phases_array_path=nx.DataPath("SyntheticVolumeDataContainer/CellData/Phases"),
    cell_quats_array_path=nx.DataPath("SyntheticVolumeDataContainer/CellData/Quats"),   # crystal_structures_array_path (nx.ArraySelectionParameter) – Enumeration representing the crystal structure for each Ensemble
    feature_ids_array_name=nx.DataPath("SyntheticVolumeDataContainer/CellData/FeatureIds"),
    # input_image_geometry_path (nx.GeometrySelectionParameter) – DataPath to input Grid Geometry
    is_periodic=True,
    misorientation_tolerance=2.0,  # deg
    # neighbor_scheme_index (nx.ChoicesParameter) – How many neighbors to use
    randomize_features=False,
    use_mask=False,
)
check_filter_result(
    nxor.EBSDSegmentFeaturesFilter(),
    result
)

In [ ]:
# WIP
result = nxor.ComputeAvgOrientationsFilter.execute(
    data_structure=ds,
    # avg_euler_angles_array_name (nx.DataObjectNameParameter) – The name of the array specifying the orientation of each Feature in Bunge convention (Z-X-Z)
    # avg_quats_array_name (nx.DataObjectNameParameter) – The name of the array specifying the average orientation of the Feature in quaternion representation
    # cell_feature_attribute_matrix_path (nx.AttributeMatrixSelectionParameter) – The path to the cell feature attribute matrix
    cell_feature_ids_array_path=nx.DataPath("SyntheticVolumeDataContainer/CellData/FeatureIds"),
    cell_phases_array_path=nx.DataPath("SyntheticVolumeDataContainer/CellData/Phases"),
    cell_quats_array_path=nx.DataPath("SyntheticVolumeDataContainer/CellData/Quats"),
    crystal_structures_array_path=nx.DataPath("SyntheticVolumeDataContainer/CellEnsembleData/CrystalStructures"),
)
check_filter_result(
    nxor.ComputeAvgOrientationsFilter(),
    result
)

In [ ]:
# WIP, step "FindAvgScalarValueForFeatures"

In [ ]:
# WIP
result = nxor.ComputeIPFColorsFilter(
    data_structure=ds,
    cell_euler_angles_array_path=nx.DataPath("SyntheticVolumeDataContainer/CellData/Eulers"),
    cell_ipf_colors_array_name="IPFColor001",
    cell_phases_array_path=nx.DataPath("SyntheticVolumeDataContainer/CellData/Phases"),
    crystal_structures_array_path=nx.DataPath("SyntheticVolumeDataContainer/CellEnsembleData/CrystalStructures"),
    mask_array_path=nx.DataPath(""),
    reference_dir=np.asarray([0, 0, 1], np.float32),
    use_mask=False
)
check_filter_result(
    nxor.ComputeIPFColorsFilter(),
    result
)

In [ ]:
# WIP repeat the above for np.asarray([0, 1, 0], np.float32),

In [ ]:
# WIP repeat the above for np.asarray([1, 0, 0], np.float32),

In [ ]:
result = nx.WriteDREAM3DFilter.execute(
    data_structure=ds,
    export_file_path=f"{root_path}/damask_to_score_dream3d.dream3d",
    write_xdmf_file=True,
)
check_filter_result(
    nx.WriteDREAM3DFilter(),
    result
)